# Verification Notebook
Verification of the MILP model

1. Unit tests on the pre-processing functions
2. Degenerate / extreme / infeasible conditions (simplified model, real constraint functions)
3. System-level checks

In [1]:
import sys
import math
import copy
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import gurobipy as gp
from gurobipy import GRB


def find_model_dir():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in ("Final_Mega_Model/ISRU_Payload_Model", "ISRU_Payload_Model"):
            if (base / rel).exists():
                return base / rel
    raise FileNotFoundError("Could not find Final_Mega_Model/ISRU_Payload_Model")


sys.path.insert(0, str(find_model_dir()))

import Constraints_creation as cc
from Linear_Func import add_log_pwl_1d
import Apollo_17.Define_Network_Vehicle_ISRU as ap_net
import Apollo_17.Define_Commodities_Supply_Demand as ap_com
import Campaign.Define_Network_Vehicle_ISRU as cp_net
import Campaign.Define_Commodities_Supply_Demand as cp_com

In [2]:
RESULTS = []


def test(test_id, title):
    def decorator(fn):
        try:
            info = fn()
            RESULTS.append((test_id, title, "PASS"))
            print(f"TEST {test_id} - PASS: {title}" + (f"\n        {info}" if info else ""))
        except AssertionError as e:
            RESULTS.append((test_id, title, "FAIL"))
            print(f"TEST {test_id} - FAIL: {title}\n        {e}")
        except Exception as e:
            RESULTS.append((test_id, title, "ERROR"))
            print(f"TEST {test_id} - ERROR: {title}\n        {e!r}")
        return fn
    return decorator

## 1. Unit Tests (Deterministic Pre-Processing)
### phi() - rocket equation

In [3]:
G0 = 9.80665
DV = {0: {0: 0, 1: 2.0}, 1: {0: 2.0, 1: 0}}      # delta-V [km/s]
ISP = np.array([0, 300])                          # vehicle 0: Isp = 0 s, vehicle 1: Isp = 300 s


@test("1.1", "phi matches the hand-derived rocket equation (dV = 2 km/s, Isp = 300 s)")
def _():
    expected = 1 - np.exp(-2000 / (300 * G0))     # = 0.4933
    got = ap_com.phi(0, 1, 1, DV, ISP, G0)
    assert np.isclose(got, expected, rtol=1e-12) and np.isclose(got, 0.4933, atol=1e-4), f"got {got}"
    return f"phi = {got:.4f}"


@test("1.2", "phi extreme cases: dV = 0 gives 0 (even for Isp = 0); Isp = 0 with dV > 0 gives 1")
def _():
    assert ap_com.phi(0, 0, 0, DV, ISP, G0) == 0, "holdover with Isp = 0"
    assert ap_com.phi(0, 0, 1, DV, ISP, G0) == 0, "holdover with Isp = 300"
    assert ap_com.phi(0, 1, 0, DV, ISP, G0) == 1, "transfer with Isp = 0"

TEST 1.1 - PASS: phi matches the hand-derived rocket equation (dV = 2 km/s, Isp = 300 s)
        phi = 0.4933
TEST 1.2 - PASS: phi extreme cases: dV = 0 gives 0 (even for Isp = 0); Isp = 0 with dV > 0 gives 1


### all_possible_outflow_arcs_()

In [4]:
def arc(arrival, duration):
    return {"ArrivalTime": arrival, "FullTravelTime": duration}


# Uniform holdover (Apollo_17): window[i][j] = departure times, holdover is a normal arc with tof[i][i]
UNI_WINDOW = {0: {0: [0, 1, 2], 1: [0]}, 1: {0: [1], 1: [0, 1]}}
UNI_TOF = {0: {0: 1, 1: 2}, 1: {0: 1, 1: 1}}
UNI_T = 3
UNI_EXPECTED = {
    0: {0: {0: arc(1, 1), 1: arc(2, 2)}, 1: {1: arc(1, 1)}},
    1: {0: {0: arc(2, 1)}, 1: {0: arc(2, 1), 1: arc(2, 1)}},
}

# Non-uniform holdover (Campaign): holdover spans the gap between consecutive departure times in window[i][i]
NU_WINDOW = {0: {0: [0, 4, 9], 1: [0, 5]}, 1: {0: [7], 1: [2, 3]}}
NU_TOF = {0: {0: 100, 1: 2}, 1: {0: 2, 1: 1}}          # tof[0][0] = 100 must be ignored
NU_T = 10
NU_EXPECTED = {
    0: {0: {0: arc(4, 4), 1: arc(2, 2)}},
    2: {1: {1: arc(3, 1)}},
    4: {0: {0: arc(9, 5)}},
    5: {0: {1: arc(7, 2)}},
    7: {1: {0: arc(9, 2)}},
}


@test("1.3", "Uniform arcs: hand-traced network matches exactly; arrival at or after T is dropped")
def _():
    got = ap_net.all_possible_outflow_arcs_uniform_holdover(window=UNI_WINDOW, tof_used=UNI_TOF, T=UNI_T)
    assert got == UNI_EXPECTED, f"\ngot      {got}\nexpected {UNI_EXPECTED}"
    assert 2 not in got, "0->0 departing at t=2 arrives at t=3=T and must not exist"


@test("1.4", "Non-uniform arcs: holdover spans the window gap (ignores tof[i][i]); last entry has no holdover; arrival >= T dropped")
def _():
    f = cp_net.all_possible_outflow_arcs_nonuniform_holdover
    got = f(window=NU_WINDOW, tof_used=NU_TOF, T=NU_T)
    assert got == NU_EXPECTED, f"\ngot      {got}\nexpected {NU_EXPECTED}"
    assert f(window={0: {1: [8]}}, tof_used={0: {1: 2}}, T=10) == {}, "arrival at t=10=T must be dropped"


@test("1.5", "Reverse mode is the exact transpose of forward mode (both generators), as the mass balance requires")
def _():
    def check(fwd, rev):
        n = lambda a: sum(1 for t in a for i in a[t] for j in a[t][i])
        assert n(fwd) == n(rev), f"arc counts differ: {n(fwd)} vs {n(rev)}"
        for t in fwd:
            for i in fwd[t]:
                for j, a in fwd[t][i].items():
                    back = rev[a["ArrivalTime"]][j][i]
                    assert back["ArrivalTime"] == t, f"arc {(t, i, j)}: reverse ArrivalTime {back['ArrivalTime']}"
                    assert back["FullTravelTime"] == -a["FullTravelTime"], f"arc {(t, i, j)}: reverse duration"

    fwd = ap_net.all_possible_outflow_arcs_uniform_holdover(window=UNI_WINDOW, tof_used=UNI_TOF, T=UNI_T)
    rev = ap_net.all_possible_outflow_arcs_uniform_holdover(          # exactly as the runner calls it
        window={i: {j: list(reversed(ts)) for j, ts in w.items()} for i, w in UNI_WINDOW.items()},
        tof_used=ap_net.reverse_tof(UNI_TOF), T=UNI_T, reverse=True)
    check(fwd, rev)

    f = cp_net.all_possible_outflow_arcs_nonuniform_holdover
    check(f(window=NU_WINDOW, tof_used=NU_TOF, T=NU_T), f(window=NU_WINDOW, tof_used=NU_TOF, T=NU_T, reverse=True))

TEST 1.3 - PASS: Uniform arcs: hand-traced network matches exactly; arrival at or after T is dropped
TEST 1.4 - PASS: Non-uniform arcs: holdover spans the window gap (ignores tof[i][i]); last entry has no holdover; arrival >= T dropped
TEST 1.5 - PASS: Reverse mode is the exact transpose of forward mode (both generators), as the mass balance requires


### consumption_matrix()

In [5]:
AP_NET, AP_VEH = ap_net.NetworkModel(campaign=False), ap_net.VehicleModel()
AP_ISRU = ap_net.ISRUModel()
AP_COM = ap_com.define_commodities(AP_ISRU)
AP_CREW, AP_CONS, AP_PROP, AP_Y = [0, 1, 6], 2, 5, 7          # crew-type columns, consumables row, propellant row, y column
AP_CARRIED = np.array(AP_VEH.carriable)

CP_NET, CP_VEH = cp_net.NetworkModel(campaign=False), cp_net.VehicleModel()
CP_ISRU = cp_net.ISRUModel()
CP_COM = cp_com.define_commodities(CP_ISRU)
CP_CREW, CP_OX, CP_KER, CP_ACTIVE, CP_MAINT, CP_Y = [0, 1, 6], 5, 9, 8, 10, 11
CP_CARRIED = np.array(CP_VEH.carriable)


def ap_matrix(i, j, v, arc_tof=None, network=AP_NET):
    return ap_com.consumption_matrix(
        i=i, j=j, v=v, commodity_names=AP_COM.commodity_names, prop_index=AP_COM.prop_index,
        crew_mass=AP_COM.crew_mass, daily_consumption=AP_COM.consumption_rate, network=network,
        vehicle_data=AP_VEH, Active_ISRU_index=AP_COM.isru_indices["active"],
        ArcTOF=network.tof[i][j] if arc_tof is None else arc_tof, commodities=AP_COM,
        days_per_year=AP_ISRU.days_per_year)


def cp_matrix(i, j, v, arc_tof=None, network=CP_NET):
    return cp_com.consumption_matrix(
        i=i, j=j, v=v, commodity_names=CP_COM.commodity_names, prop_index=CP_COM.prop_index,
        crew_mass=CP_COM.crew_mass, daily_consumption=CP_COM.consumption_rate, network=network,
        vehicle_data=CP_VEH, Active_ISRU_index=CP_ACTIVE,
        ArcTOF=network.tof[i][j] if arc_tof is None else arc_tof, commodities=CP_COM,
        days_per_year=CP_ISRU.days_per_year)


@test("1.6", "Apollo propellant row (LEO->LLO, dV = 4.04): hand-derived rocket equation; carrier's Isp used; carried structure mass charged")
def _():
    for v in (1, 3):                                          # S-IVB (Isp 421) and Service module (Isp 314)
        p = 1 - np.exp(-4040 / (AP_VEH.isp[v] * AP_NET.g0))
        expected = np.zeros(12)
        expected[AP_CREW] = -100 * p                          # crew mass 100 kg
        expected[[2, 3, 4]] = -p                              # consumables, equipment, samples (1 kg per kg)
        expected[AP_PROP] = 1 - p
        expected[AP_Y] = -AP_VEH.structure_mass[v] * p        # own structure
        expected[8:12] = -AP_VEH.structure_mass[AP_CARRIED] * p   # carried CM, SM, LM-d, LM-a
        got = ap_matrix(1, 2, v)[AP_PROP]
        assert np.allclose(got, expected, rtol=1e-12), f"v={v}\ngot      {got}\nexpected {expected}"


@test("1.7", "Apollo consumables: rate x crew x ArcTOF (not tof[i][j]); PAC->LEO and Earth holdover are the identity")
def _():
    rate = 124 / 30                                           # kg/crew/day
    m = ap_matrix(3, 3, 4, arc_tof=5)                         # 5-day lunar-surface holdover (network tof says 1)
    expected = np.zeros(12)
    expected[AP_CREW] = -rate * 5
    expected[AP_CONS] = 1
    assert np.allclose(m[AP_CONS], expected), f"got {m[AP_CONS]}"
    assert np.array_equal(m[AP_PROP], np.eye(12)[AP_PROP]), "a holdover must not burn propellant"
    assert np.array_equal(ap_matrix(0, 1, 0), np.eye(12)), "PAC->LEO should be the identity"
    earth = SimpleNamespace(delta_v={0: {0: 0}}, g0=AP_NET.g0, tof={0: {0: 5}})
    assert np.array_equal(ap_matrix(0, 0, 0, network=earth), np.eye(12)), "Earth holdover should be the identity"


@test("1.8", "Apollo extreme: Isp = 0 vehicle (Command module) gives phi = 1: propellant self-retention 0, own structure fully charged")
def _():
    v = 2
    assert AP_VEH.isp[v] == 0
    m = ap_matrix(2, 3, v)                                    # LLO -> lunar surface, dV = 1.87
    assert m[AP_PROP, AP_PROP] == 0, f"self-retention {m[AP_PROP, AP_PROP]}"
    assert m[AP_PROP, AP_Y] == -AP_VEH.structure_mass[v], f"own structure term {m[AP_PROP, AP_Y]}"
    assert np.all(m[AP_PROP, AP_CREW] == -100) and np.all(m[AP_PROP, [2, 3, 4]] == -1)


@test("1.9", "Campaign: oxygen + kerosene rows add up to the single-propellant rocket equation; split follows prop_percentages; boil-off only off Earth/PAC")
def _():
    v, boil = 0, CP_COM.oxygen_boiloff
    p0, p1 = CP_COM.prop_percentages
    p = 1 - np.exp(-4040 / (CP_VEH.isp[v] * CP_NET.g0))
    m = cp_matrix(1, 2, v)
    ox, ker = m[CP_OX] / (1 - boil), m[CP_KER]                # divide the oxygen boil-off out
    expected = np.zeros(14)
    expected[CP_CREW] = -100 * p
    expected[[2, 3, 4, 7, 8, 10]] = -p                        # 1 kg per kg commodities (incl. ISRU and maintenance mass)
    expected[[CP_OX, CP_KER]] = 1 - p
    expected[CP_Y] = -CP_VEH.structure_mass[v] * p
    expected[12:14] = -CP_VEH.structure_mass[CP_CARRIED] * p
    assert np.allclose(ox + ker, expected, rtol=1e-12), f"\ngot      {ox + ker}\nexpected {expected}"
    assert np.isclose(ox[CP_OX], 1 - p * p0) and np.isclose(ker[CP_KER], 1 - p * p1), "split differs from prop_percentages"
    # hypothetical burn on PAC->LEO: no boil-off applied
    pac = cp_matrix(0, 1, v, network=SimpleNamespace(delta_v={0: {1: 4.04}}, g0=CP_NET.g0, tof={0: {1: 1}}))
    assert np.isclose(pac[CP_OX, CP_OX], 1 - p * p0), "boil-off wrongly applied on PAC->LEO"


@test("1.10", "Campaign maintenance: 1% of own and carried structure mass per arc; active ISRU 10%/yr scaled by ArcTOF; zero on PAC->LEO")
def _():
    v = 0
    m = cp_matrix(3, 3, v, arc_tof=30)
    assert np.isclose(m[CP_MAINT, CP_Y], -0.01 * CP_VEH.structure_mass[v]), "own structure maintenance"
    assert np.allclose(m[CP_MAINT, 12:14], -0.01 * CP_VEH.structure_mass[CP_CARRIED]), "carried structure maintenance"
    assert np.isclose(m[CP_MAINT, CP_ACTIVE], -(0.1 / 365) * 30), f"active ISRU maintenance {m[CP_MAINT, CP_ACTIVE]}"
    assert np.array_equal(cp_matrix(0, 1, v)[CP_MAINT], np.eye(14)[CP_MAINT]), "maintenance on PAC->LEO"

Network model is valid
Network model is valid
TEST 1.6 - PASS: Apollo propellant row (LEO->LLO, dV = 4.04): hand-derived rocket equation; carrier's Isp used; carried structure mass charged
TEST 1.7 - PASS: Apollo consumables: rate x crew x ArcTOF (not tof[i][j]); PAC->LEO and Earth holdover are the identity
TEST 1.8 - PASS: Apollo extreme: Isp = 0 vehicle (Command module) gives phi = 1: propellant self-retention 0, own structure fully charged
TEST 1.9 - PASS: Campaign: oxygen + kerosene rows add up to the single-propellant rocket equation; split follows prop_percentages; boil-off only off Earth/PAC
TEST 1.10 - PASS: Campaign maintenance: 1% of own and carried structure mass per arc; active ISRU 10%/yr scaled by ArcTOF; zero on PAC->LEO


### create_concurrency_constraint() and create_sc_design_parameters()

In [6]:
@test("1.11", "Concurrency rows: propellant removed from the payload row and unit vector in the propellant row (Apollo and 2-propellant Campaign)")
def _():
    H = cc.create_concurrency_constraint(AP_COM.mass_conversion, AP_COM.prop_index)
    assert np.array_equal(H, [[100, 100, 1, 1, 1, 0, 100], [0, 0, 0, 0, 0, 1, 0]]), f"Apollo:\n{H}"
    H = cc.create_concurrency_constraint(CP_COM.mass_conversion, CP_COM.prop_index)
    assert np.array_equal(H, [[100, 100, 1, 1, 1, 0, 100, 1, 1, 0, 1],
                              [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0]]), f"Campaign:\n{H}"


@test("1.12", "Design parameters e[v] = [own cap, caps of carriable types]: Apollo hand values and a non-trivial carriable pattern")
def _():
    e = cc.create_sc_design_parameters(AP_VEH)
    assert e.shape == (6, 2, 5), f"shape {e.shape}"
    own_pay, own_prop = [0, 0, 524, 60, 500, 250], [452045, 107725, 0, 18413, 8917, 2358]
    for v in range(6):
        assert list(e[v][0]) == [own_pay[v], 524, 60, 500, 250], f"v={v} payload {e[v][0]}"
        assert list(e[v][1]) == [own_prop[v], 0, 18413, 8917, 2358], f"v={v} propellant {e[v][1]}"
    veh = SimpleNamespace(carriable=[True, False, True], structure_mass=np.array([1, 1, 1]),
                          payload_cap=np.array([10, 20, 30]), propellant_cap=np.array([1, 2, 3]))
    e = cc.create_sc_design_parameters(veh)
    for v in range(3):
        assert list(e[v][0]) == [veh.payload_cap[v], 10, 30] and list(e[v][1]) == [veh.propellant_cap[v], 1, 3], f"v={v}"

TEST 1.11 - PASS: Concurrency rows: propellant removed from the payload row and unit vector in the propellant row (Apollo and 2-propellant Campaign)
TEST 1.12 - PASS: Design parameters e[v] = [own cap, caps of carriable types]: Apollo hand values and a non-trivial carriable pattern


### ISRU productivity function

In [7]:
@test("1.13", "ISRUfunc is exactly 0 below 400 kg and matches the hand formula at 2000 kg")
def _():
    for x in (0, 300, 399.999):
        assert ap_net.ISRUfunc(x) == 0, f"x={x}: {ap_net.ISRUfunc(x)}"
    expected = -0.438 + 6.9623 * (1 - math.exp(-2000 / 812.1563)) + 2.0173 * (1 - math.exp(-2000 / 3967.2644))
    assert np.isclose(ap_net.ISRUfunc(2000), expected, rtol=1e-9), f"{ap_net.ISRUfunc(2000)} vs {expected}"
    assert np.isclose(ap_net.ISRU_total_annual_output(2000), 2000 * expected), "total output = mass x productivity"

TEST 1.13 - PASS: ISRUfunc is exactly 0 below 400 kg and matches the hand formula at 2000 kg


### add_log_pwl_1d() - logarithmic piecewise-linear model

In [8]:
def pwl_range(func, x_lb, x_ub, n_seg, x_fix, z_fix=None, x_bounds=None, **kw):
    """(z_min, z_max) of the PWL model at x = x_fix, or None if infeasible."""
    m = gp.Model("pwl")
    m.Params.OutputFlag = 0
    lo, hi = x_bounds if x_bounds else (x_fix, x_fix)
    x = m.addVar(lb=lo, ub=hi, name="x")
    z = m.addVar(lb=z_fix, ub=z_fix, name="z") if z_fix is not None else None
    info = add_log_pwl_1d(m, func, x_lb, x_ub, n_seg, name="t", x_var=x, z_var=z, **kw)
    out = []
    for sense in (GRB.MINIMIZE, GRB.MAXIMIZE):
        m.setObjective(info["z_var"], sense)
        m.optimize()
        if m.Status != GRB.OPTIMAL:
            assert m.Status in (GRB.INFEASIBLE, GRB.INF_OR_UNBD), f"unexpected status {m.Status}"
            return None
        out.append(m.ObjVal)
    return tuple(out)


SQ = lambda x: x ** 2


@test("1.14", "PWL: exact at breakpoints, linear between them, and cannot cheat by mixing non-adjacent breakpoints (f = x^2, 10 segments)")
def _():
    for x, expected in ((3, 9), (3.5, 12.5), (5.5, 30.5)):    # 5.5 would be 55 if breakpoints 0 and 10 could be mixed
        lo, hi = pwl_range(SQ, 0, 10, 10, x)
        assert np.isclose(lo, expected) and np.isclose(hi, expected), f"x={x}: [{lo}, {hi}] vs {expected}"
    f = lambda x: x ** 2 + 3                                  # a non power-of-two segment count (unused Gray codes)
    bp = np.linspace(0, 10, 7)
    for x in (0, 1.3, 5, 7.7, 10):
        lo, hi = pwl_range(f, 0, 10, 6, x)
        assert np.isclose(lo, np.interp(x, bp, f(bp))) and np.isclose(hi, np.interp(x, bp, f(bp))), f"n=6, x={x}"

Set parameter Username
Set parameter LicenseID to value 2739971
Academic license - for non-commercial use only - expires 2026-11-17
TEST 1.14 - PASS: PWL: exact at breakpoints, linear between them, and cannot cheat by mixing non-adjacent breakpoints (f = x^2, 10 segments)


### demand_supply()

In [9]:
@test("1.15", "Apollo demand/supply: shapes, hand-checked entries (negative = demand), crew-interim and crew-return cancel")
def _():
    names = AP_COM.commodity_names
    c = {n: k for k, n in enumerate(names)}
    V = AP_VEH.number_vehicle_types
    D, d = ap_com.demand_supply(AP_NET, len(names), V)
    assert len(D) == 4 and len(D[0]) == AP_NET.T and D[0][0].shape == (7,), "D must be [node][time][commodity]"
    assert len(d) == 4 and len(d[0]) == V and len(d[0][0]) == AP_NET.T, "d must be [node][vehicle][time]"
    expected = {
        (3, 5, "crew"): -2, (2, 4, "crew"): -1,                                       # 3 crew delivered
        (3, 5, "crew_interim"): 2, (2, 4, "crew_interim"): 1,
        (3, 6, "crew_interim"): -2, (2, 7, "crew_interim"): -1,
        (3, 6, "crew_return"): 2, (2, 7, "crew_return"): 1, (0, 11, "crew_return"): -3,
        (3, 5, "equipment"): -420, (0, 11, "samples"): -110,
    }
    for (i, t, name), value in expected.items():
        assert D[i][t][c[name]] == value, f"D[{i}][{t}][{name}] = {D[i][t][c[name]]}, expected {value}"
    finite = lambda k: sum(D[i][t][k] for i in range(4) for t in range(AP_NET.T) if abs(D[i][t][k]) < 1e14)
    assert finite(c["crew_interim"]) == 0 and finite(c["crew_return"]) == 0, "supply and demand of interim/return crew must cancel"
    assert [d[0][v][0] for v in range(V)] == [0, 1, 1, 1, 1, 1], "one of each vehicle (except Saturn V second stage) at Earth, t=0"

TEST 1.15 - PASS: Apollo demand/supply: shapes, hand-checked entries (negative = demand), crew-interim and crew-return cancel


### Summary Unit Tests

In [10]:
for tid, title, status in RESULTS:
    print(f"{tid:<5} {status:<6} {title}")
counts = {s: sum(r[2] == s for r in RESULTS) for s in ("PASS", "FAIL", "ERROR")}
print(f"\n{counts['PASS']} passed, {counts['FAIL']} failed, {counts['ERROR']} errors out of {len(RESULTS)} tests")

1.1   PASS   phi matches the hand-derived rocket equation (dV = 2 km/s, Isp = 300 s)
1.2   PASS   phi extreme cases: dV = 0 gives 0 (even for Isp = 0); Isp = 0 with dV > 0 gives 1
1.3   PASS   Uniform arcs: hand-traced network matches exactly; arrival at or after T is dropped
1.4   PASS   Non-uniform arcs: holdover spans the window gap (ignores tof[i][i]); last entry has no holdover; arrival >= T dropped
1.5   PASS   Reverse mode is the exact transpose of forward mode (both generators), as the mass balance requires
1.6   PASS   Apollo propellant row (LEO->LLO, dV = 4.04): hand-derived rocket equation; carrier's Isp used; carried structure mass charged
1.7   PASS   Apollo consumables: rate x crew x ArcTOF (not tof[i][j]); PAC->LEO and Earth holdover are the identity
1.8   PASS   Apollo extreme: Isp = 0 vehicle (Command module) gives phi = 1: propellant self-retention 0, own structure fully charged
1.9   PASS   Campaign: oxygen + kerosene rows add up to the single-propellant rocket equat

## 2. Extreme, Degenerate and Infeasible Conditions


In [11]:
from Dataclasses import NetworkData, VehicleData
from Variable_Creation import create_commodity_flow, create_sc_commodity_flow
import Apollo_17.Define_Cost_Func as ap_cost

TINY_T = 6
BIG = 1e9            # "unlimited" supply at Earth


def build_tiny_model(demand_kg=100.0, demand_time=2, dv=3.0, isp=300.0, n_vehicles=2,
                     payload_cap=2000.0, prop_cap=20000.0, structure=1000.0):
    """Earth (0) -> destination (1) in 2 days, one non-carriable vehicle type, equipment demand at the destination."""
    net = NetworkData()
    net.g0 = 9.80665
    net.T = TINY_T
    net.connections = {0: [1], 1: [0, 1]}
    net.tof = {0: {1: 2}, 1: {0: 2, 1: 1}}
    net.delta_v = {0: {1: dv}, 1: {0: dv, 1: 0}}
    net.node_windows = {i: {j: [t for t in range(TINY_T) if t + net.tof[i][j] < TINY_T] for j in net.connections[i]}
                        for i in net.connections}
    veh = VehicleData(number_vehicle_types=1, vehicle_type_names=["Tiny"], structure_mass=np.array([structure]),
                      isp=np.array([isp]), payload_cap=np.array([payload_cap]), propellant_cap=np.array([prop_cap]),
                      carriable=[False], max_carried=[0], sc_vtype=GRB.INTEGER)
    com = ap_com.define_commodities(ap_net.ISRUModel())          # same 7 commodities as Apollo_17
    equip, prop = com.commodity_names.index("equipment"), com.prop_index[0]

    # demand [node][time][commodity] (negative = demand, positive = supply) and vehicle supply [node][vehicle][time]
    D = [[np.zeros(len(com.commodity_names)) for _ in range(TINY_T)] for _ in net.connections]
    for t in range(TINY_T):
        D[0][t][equip] = BIG
        D[0][t][prop] = BIG
    D[1][demand_time][equip] = -demand_kg
    d = [[[n_vehicles if (i == 0 and t == 0) else 0 for t in range(TINY_T)]] for i in net.connections]

    arcs = ap_net.all_possible_outflow_arcs_uniform_holdover(window=net.node_windows, tof_used=net.tof, T=net.T)
    rev = ap_net.all_possible_outflow_arcs_uniform_holdover(
        window={i: {j: list(reversed(ts)) for j, ts in w.items()} for i, w in net.node_windows.items()},
        tof_used=ap_net.reverse_tof(net.tof), T=net.T, reverse=True)

    m = gp.Model("tiny")
    m.Params.OutputFlag = 0
    m.Params.DualReductions = 0              # report INFEASIBLE rather than INF_OR_UNBD
    flow = lambda direction: create_commodity_flow(m, 1, com, arcs, net.connections, direction, "Classic")
    sc = lambda direction: create_sc_commodity_flow(m, 1, veh.sc_vtype, arcs, net.connections, direction)
    payload = lambda direction: create_commodity_flow(m, 1, veh, arcs, net.connections, direction, "SC_payload")
    ctx = {"model": m, "network": net, "vehicle_data": veh, "V": 1, "Commodities": com,
           "isru_config": ap_net.ISRUModel(), "isru_production_model": ap_net.ISRU_total_annual_output,
           "Demands": D, "V_Demands": d, "connections": net.connections, "all_arcs": arcs, "rev_arcs": rev,
           "x_outflow": flow("out"), "x_inflow": flow("in"), "y_outflow": sc("out"), "y_inflow": sc("in"),
           "scpayload_outflow": payload("out"), "scpayload_inflow": payload("in")}
    m.update()

    cc.add_mass_balance_constraints(m, ctx)
    cc.add_arc_transformation_constraints(m, ctx, ap_com.consumption_matrix)
    cc.add_concurrency_constraints(m, ctx)
    # add_SCP_concurrency_constraint is skipped: it only limits carried spacecraft, and this vehicle is not carriable
    # (with no carriable type the function compares a plain 0 with a Gurobi expression and fails).
    cc.add_time_window_constraints(m, ctx)
    m.setObjective(ap_cost.set_initial_mass_objective(m, ctx, start_node=0, end_node=1), GRB.MINIMIZE)
    m.optimize()
    return m, ctx


def vehicles_launched(ctx):
    return sum(ctx["y_outflow"][0][0][1][t][0].X for t in ctx["network"].node_windows[0][1])


def mass_launched_by_hand(demand_kg, structure, n_vehicles):
    """Dry-mass estimate when propellant is negligible: payload + n structures."""
    return demand_kg + n_vehicles * structure

In [12]:
@test("2.1", "Zero demand: model is optimal with objective 0 and nothing moves")
def _():
    m, ctx = build_tiny_model(demand_kg=0.0)
    assert m.Status == GRB.OPTIMAL, f"status {m.Status}"
    assert abs(m.ObjVal) < 1e-9, f"objective {m.ObjVal}"
    assert all(abs(v.X) < 1e-9 for v in m.getVars()), "some variable is non-zero although nothing is demanded"


@test("2.2", "Very large Isp (phi ~ 0): objective = payload + vehicle structure, propellant negligible (hand: 100 + 1000 kg)")
def _():
    m, ctx = build_tiny_model(demand_kg=100.0, isp=1e9)
    assert m.Status == GRB.OPTIMAL, f"status {m.Status}"
    assert vehicles_launched(ctx) == 1, f"{vehicles_launched(ctx)} vehicles launched"
    assert np.isclose(m.ObjVal, mass_launched_by_hand(100, 1000, 1), atol=0.01), f"objective {m.ObjVal}, hand value 1100"
    return f"objective = {m.ObjVal:.4f} kg"


@test("2.3", "Demand just over the payload capacity (2000 kg): second vehicle is launched, objective jumps by one structure mass")
def _():
    below, ctx_b = build_tiny_model(demand_kg=1900.0, isp=1e9, n_vehicles=3)
    above, ctx_a = build_tiny_model(demand_kg=2100.0, isp=1e9, n_vehicles=3)
    assert below.Status == GRB.OPTIMAL and above.Status == GRB.OPTIMAL, f"status {below.Status}, {above.Status}"
    assert vehicles_launched(ctx_b) == 1 and vehicles_launched(ctx_a) == 2, \
        f"vehicles launched: {vehicles_launched(ctx_b)} (1900 kg), {vehicles_launched(ctx_a)} (2100 kg)"
    assert np.isclose(below.ObjVal, mass_launched_by_hand(1900, 1000, 1), atol=0.1), f"{below.ObjVal} vs 2900"
    assert np.isclose(above.ObjVal, mass_launched_by_hand(2100, 1000, 2), atol=0.1), f"{above.ObjVal} vs 4100"
    return f"objective {below.ObjVal:.1f} kg -> {above.ObjVal:.1f} kg"


@test("2.4", "Demand that cannot arrive in time (earliest arrival is t = 2, demand at t = 1): infeasible, feasible baseline at t = 2")
def _():
    base, _ = build_tiny_model(demand_time=2)
    assert base.Status == GRB.OPTIMAL, f"baseline status {base.Status}"
    m, _ = build_tiny_model(demand_time=1)
    assert m.Status == GRB.INFEASIBLE, f"status {m.Status}"


@test("2.5", "No vehicle supplied: cargo cannot move itself, infeasible (feasible baseline with 2 vehicles)")
def _():
    base, _ = build_tiny_model(n_vehicles=2)
    assert base.Status == GRB.OPTIMAL, f"baseline status {base.Status}"
    m, _ = build_tiny_model(n_vehicles=0)
    assert m.Status == GRB.INFEASIBLE, f"status {m.Status}"


@test("2.6", "Impossible burn: dV = 25 km/s needs far more propellant than the tank holds, infeasible (feasible baseline at dV = 3)")
def _():
    base, _ = build_tiny_model(dv=3.0)
    assert base.Status == GRB.OPTIMAL, f"baseline status {base.Status}"
    m, _ = build_tiny_model(dv=25.0)
    phi_25 = 1 - np.exp(-25000 / (300 * 9.80665))
    needed = phi_25 / (1 - phi_25) * (1000 + 100)            # propellant to deliver structure + payload, from the rocket equation
    assert needed > 20000, "test setup: hand-computed propellant need should exceed the 20000 kg tank"
    assert m.Status == GRB.INFEASIBLE, f"status {m.Status}"
    return f"hand-computed propellant need {needed:,.0f} kg vs tank 20,000 kg"

TEST 2.1 - PASS: Zero demand: model is optimal with objective 0 and nothing moves
TEST 2.2 - PASS: Very large Isp (phi ~ 0): objective = payload + vehicle structure, propellant negligible (hand: 100 + 1000 kg)
        objective = 1100.0003 kg
TEST 2.3 - PASS: Demand just over the payload capacity (2000 kg): second vehicle is launched, objective jumps by one structure mass
        objective 2900.0 kg -> 4100.0 kg
TEST 2.4 - PASS: Demand that cannot arrive in time (earliest arrival is t = 2, demand at t = 1): infeasible, feasible baseline at t = 2
TEST 2.5 - PASS: No vehicle supplied: cargo cannot move itself, infeasible (feasible baseline with 2 vehicles)
TEST 2.6 - PASS: Impossible burn: dV = 25 km/s needs far more propellant than the tank holds, infeasible (feasible baseline at dV = 3)
        hand-computed propellant need 5,392,376 kg vs tank 20,000 kg


## 3. System-Level Checks


In [13]:
import time
import Campaign.Define_Cost_Func as cp_cost
from collections import defaultdict

CASES = {
    "Apollo 17": SimpleNamespace(net=ap_net, com=ap_com, cost=ap_cost, uniform=True),
    "Campaign": SimpleNamespace(net=cp_net, com=cp_com, cost=cp_cost, uniform=False),
}


def build_full_model(case, isru_enabled=None, edit_demand=None):
    sc = CASES[case]
    net, veh, isru = sc.net.NetworkModel(campaign=False), sc.net.VehicleModel(), sc.net.ISRUModel()
    if isru_enabled is not None:
        isru.enabled = isru_enabled
    com = sc.com.define_commodities(isru)
    V = veh.number_vehicle_types
    D, d = sc.com.demand_supply(net, len(com.commodity_names), V)
    if edit_demand:
        edit_demand(D, d, com)

    if sc.uniform:
        gen = sc.net.all_possible_outflow_arcs_uniform_holdover
        arcs = gen(window=net.node_windows, tof_used=net.tof, T=net.T)
        rev = gen(window={i: {j: list(reversed(ts)) for j, ts in w.items()} for i, w in net.node_windows.items()},
                  tof_used=sc.net.reverse_tof(net.tof), T=net.T, reverse=True)
    else:
        gen = sc.net.all_possible_outflow_arcs_nonuniform_holdover
        arcs = gen(window=net.node_windows, tof_used=net.tof, T=net.T)
        rev = gen(window=net.node_windows, tof_used=net.tof, T=net.T, reverse=True)

    m = gp.Model(case)
    m.Params.OutputFlag = 0
    flow = lambda dr: create_commodity_flow(m, V, com, arcs, net.connections, dr, "Classic")
    spacecraft = lambda dr: create_sc_commodity_flow(m, V, veh.sc_vtype, arcs, net.connections, dr)
    payload = lambda dr: create_commodity_flow(m, V, veh, arcs, net.connections, dr, "SC_payload")
    ctx = {"model": m, "network": net, "vehicle_data": veh, "V": V, "Commodities": com, "isru_config": isru,
           "isru_production_model": sc.net.ISRU_total_annual_output, "Demands": D, "V_Demands": d,
           "connections": net.connections, "all_arcs": arcs, "rev_arcs": rev,
           "x_outflow": flow("out"), "x_inflow": flow("in"), "y_outflow": spacecraft("out"), "y_inflow": spacecraft("in"),
           "scpayload_outflow": payload("out"), "scpayload_inflow": payload("in")}
    m.update()

    cc.add_mass_balance_constraints(m, ctx)
    cc.add_arc_transformation_constraints(m, ctx, sc.com.consumption_matrix)
    if isru.enabled:
        cc.add_ISRU_negation_constraint(m, ctx)
    cc.add_concurrency_constraints(m, ctx)
    cc.add_SCP_concurrency_constraint(m, ctx)
    cc.add_time_window_constraints(m, ctx)
    m.setObjective(sc.cost.set_initial_mass_objective(m, ctx, start_node=0, end_node=1), GRB.MINIMIZE)
    m.update()
    m.optimize()
    return m, ctx


SOLVED = {}
for name, kwargs in [("Apollo 17", {}), ("Campaign", {}), ("Campaign, ISRU off", {"isru_enabled": False})]:
    t0 = time.time()
    SOLVED[name] = build_full_model(name.split(",")[0], **kwargs)
    m_, _ = SOLVED[name]
    print(f"{name:<20} status {m_.Status}  objective {m_.ObjVal:,.1f} kg  ({m_.NumVars} variables, {time.time() - t0:.1f} s)")

# helpers to read the solution
val = lambda a: np.array([v.X for v in np.ravel(a)])


def all_arcs(ctx):
    for t in ctx["all_arcs"]:
        for i in ctx["all_arcs"][t]:
            for j, a in ctx["all_arcs"][t][i].items():
                yield t, i, j, a


def arc_vectors(ctx, v, i, j, t, direction):
    """[commodities..., y, carried spacecraft...] at the start (out) or end (in) of an arc."""
    return np.concatenate([val(ctx[f"x_{direction}flow"][v][i][j][t]), val(ctx[f"y_{direction}flow"][v][i][j][t]),
                           val(ctx[f"scpayload_{direction}flow"][v][i][j][t])])

Network model is valid
Apollo 17            status 2  objective 107,150.8 kg  (13680 variables, 0.9 s)
Network model is valid
Campaign             status 2  objective 1,298,790.5 kg  (3324 variables, 3.8 s)
Network model is valid
Campaign, ISRU off   status 2  objective 1,337,333.6 kg  (2016 variables, 0.8 s)


In [14]:
@test("3.1", "Both cases solve to optimality with integral integer variables")
def _():
    for name in ("Apollo 17", "Campaign"):
        m, ctx = SOLVED[name]
        assert m.Status == GRB.OPTIMAL, f"{name}: status {m.Status}"
        worst = max((abs(v.X - round(v.X)) for v in m.getVars() if v.VType in (GRB.INTEGER, GRB.BINARY)), default=0)
        assert worst < 1e-5, f"{name}: integer variable off by {worst}"


@test("3.2", "Arc transformation audit: recomputed inflow = consumption_matrix @ outflow on every arc (both cases)")
def _():
    report = []
    for name in ("Apollo 17", "Campaign"):
        m, ctx = SOLVED[name]
        sc, com, veh, isru = CASES[name], ctx["Commodities"], ctx["vehicle_data"], ctx["isru_config"]
        worst = 0.0
        for t, i, j, a in all_arcs(ctx):
            for v in range(ctx["V"]):
                out, inn = arc_vectors(ctx, v, i, j, t, "out"), arc_vectors(ctx, v, i, j, t, "in")
                M = sc.com.consumption_matrix(
                    i=i, j=j, v=v, commodity_names=com.commodity_names, prop_index=com.prop_index,
                    crew_mass=com.crew_mass, daily_consumption=com.consumption_rate, network=ctx["network"],
                    vehicle_data=veh, Active_ISRU_index=com.isru_indices["active"], ArcTOF=a["FullTravelTime"],
                    commodities=com, days_per_year=isru.days_per_year)
                residual = inn - M @ out
                if isru.enabled and cc.is_eligible_isru_arc(i, j, isru):
                    # lunar holdover: the oxygen row also receives the ISRU production credit, which may only add propellant
                    assert residual[com.prop_index[0]] >= -1e-4, f"{name}: negative ISRU credit on arc {(t, i, j)}, v={v}"
                    residual[com.prop_index[0]] = 0
                worst = max(worst, np.abs(residual).max())
                assert np.abs(residual).max() < 1e-4, f"{name}: arc {(t, i, j)}, v={v}: residual {np.abs(residual).max()}"
        report.append(f"{name}: largest residual {worst:.1e}")
    return "; ".join(report)


@test("3.3", "Node balance and capacity audit: balances, payload/propellant capacity and non-negativity hold (both cases)")
def _():
    report = []
    for name in ("Apollo 17", "Campaign"):
        m, ctx = SOLVED[name]
        com, veh, D, d = ctx["Commodities"], ctx["vehicle_data"], ctx["Demands"], ctx["V_Demands"]
        isru = ctx["isru_config"]
        n_com, V = len(com.commodity_names), ctx["V"]
        carried = [v for v, c in enumerate(veh.carriable) if c]
        out, inn = defaultdict(lambda: np.zeros(n_com)), defaultdict(lambda: np.zeros(n_com))
        v_out, v_in = defaultdict(lambda: np.zeros(V)), defaultdict(lambda: np.zeros(V))
        for t, i, j, a in all_arcs(ctx):
            for v in range(V):
                x_o, x_i = val(ctx["x_outflow"][v][i][j][t]), val(ctx["x_inflow"][v][i][j][t])
                y_o, y_i = val(ctx["y_outflow"][v][i][j][t])[0], val(ctx["y_inflow"][v][i][j][t])[0]
                p_o, p_i = val(ctx["scpayload_outflow"][v][i][j][t]), val(ctx["scpayload_inflow"][v][i][j][t])
                out[(i, t)] += x_o
                inn[(j, a["ArrivalTime"])] += x_i
                v_out[(i, t)][v] += y_o
                v_in[(j, a["ArrivalTime"])][v] += y_i
                for c, n_o, n_i in zip(carried, p_o, p_i):          # carried spacecraft count as spacecraft of their type
                    v_out[(i, t)][c] += n_o
                    v_in[(j, a["ArrivalTime"])][c] += n_i
                # capacity (re-derived from the vehicle data)
                cap_pay = veh.payload_cap[v] * y_o + sum(veh.payload_cap[c] * n for c, n in zip(carried, p_o))
                cap_prop = veh.propellant_cap[v] * y_o + sum(veh.propellant_cap[c] * n for c, n in zip(carried, p_o))
                prop_mass = sum(x_o[k] for k in com.prop_index)
                pay_mass = sum(com.mass_conversion[k] * x_o[k] for k in range(n_com) if k not in com.prop_index)
                assert pay_mass <= cap_pay + 1e-4, f"{name}: arc {(t, i, j)}, v={v}: payload {pay_mass} > capacity {cap_pay}"
                assert prop_mass <= cap_prop + 1e-4, f"{name}: arc {(t, i, j)}, v={v}: propellant {prop_mass} > capacity {cap_prop}"
        for (i, t) in set(out) | set(inn):
            inflow = inn[(i, t)].copy()
            if isru.enabled and cc.is_eligible_isru_arc(i, i, isru):
                # ISRU node: packaged ISRU may be activated here, so active out <= active in + packaged in,
                # and active + packaged together obey one combined balance
                act, pack = com.isru_indices["active"], com.isru_indices["packaged"]
                combined = out[(i, t)][act] + out[(i, t)][pack] - inn[(i, t)][act] - inn[(i, t)][pack]
                assert combined <= D[i][t][act] + D[i][t][pack] + 1e-4, f"{name}: combined ISRU balance violated at node {i}, t={t}"
                inflow[act] += inn[(i, t)][pack]
            net_flow = out[(i, t)] - inflow
            assert np.all(net_flow <= D[i][t] + 1e-4), f"{name}: commodity balance violated at node {i}, t={t}"
            assert np.all(v_out[(i, t)] - v_in[(i, t)] <= [d[i][v][t] for v in range(V)] + 1e-4 * np.ones(V)), \
                f"{name}: spacecraft balance violated at node {i}, t={t}"
        # every demand (negative entry) must sit on a node-time that actually has a balance constraint
        active = set(out) | set(inn)
        for i in range(len(D)):
            for t in range(len(D[i])):
                if np.any(D[i][t] < 0):
                    assert (i, t) in active, f"{name}: demand at node {i}, t={t} is not covered by any arc"
        assert min(v.X for v in m.getVars()) >= -1e-6, f"{name}: a variable is negative"
        report.append(f"{name}: {len(active)} node-times checked")
    return "; ".join(report)


@test("3.4", "Objective recomputation: launched mass from the solved values equals the solver's objective (both cases)")
def _():
    report = []
    for name in ("Apollo 17", "Campaign"):
        m, ctx = SOLVED[name]
        com, veh = ctx["Commodities"], ctx["vehicle_data"]
        carried = [v for v, c in enumerate(veh.carriable) if c]
        launched = 0.0
        for t, i, j, a in all_arcs(ctx):
            if i != 0 or j == 0:                                  # everything leaving Earth surface
                continue
            for v in range(ctx["V"]):
                x_o = val(ctx["x_outflow"][v][i][j][t])
                y_o = val(ctx["y_outflow"][v][i][j][t])[0]
                p_o = val(ctx["scpayload_outflow"][v][i][j][t])
                launched += np.dot(com.mass_conversion, x_o) + veh.structure_mass[v] * y_o \
                    + sum(veh.structure_mass[c] * n for c, n in zip(carried, p_o))
        assert np.isclose(launched, m.ObjVal, rtol=1e-9, atol=1e-4), f"{name}: recomputed {launched:,.4f} vs objective {m.ObjVal:,.4f}"
        report.append(f"{name}: {launched:,.1f} kg")
    return "; ".join(report)


@test("3.5", "Apollo 17: more lunar equipment demand raises the objective, less lowers it")
def _():
    eq = ap_com.define_commodities(ap_net.ISRUModel()).commodity_names.index("equipment")
    base = SOLVED["Apollo 17"][0].ObjVal

    def solve_with(amount):
        def edit(D, d, com):
            assert D[3][5][eq] == -420, "test assumes the original 420 kg equipment demand at the lunar surface, t = 5"
            D[3][5][eq] = -amount
        m, _ = build_full_model("Apollo 17", edit_demand=edit)
        assert m.Status == GRB.OPTIMAL, f"status {m.Status} for {amount} kg"
        return m.ObjVal

    more, less = solve_with(520), solve_with(320)
    assert less < base < more, f"objective: {less:,.1f} (320 kg) < {base:,.1f} (420 kg) < {more:,.1f} (520 kg) violated"
    return f"objective: {less:,.1f} (320 kg) < {base:,.1f} (420 kg) < {more:,.1f} (520 kg)"

TEST 3.1 - PASS: Both cases solve to optimality with integral integer variables
TEST 3.2 - PASS: Arc transformation audit: recomputed inflow = consumption_matrix @ outflow on every arc (both cases)
        Apollo 17: largest residual 7.2e-10; Campaign: largest residual 4.5e-08
TEST 3.3 - PASS: Node balance and capacity audit: balances, payload/propellant capacity and non-negativity hold (both cases)
        Apollo 17: 48 node-times checked; Campaign: 27 node-times checked
TEST 3.4 - PASS: Objective recomputation: launched mass from the solved values equals the solver's objective (both cases)
        Apollo 17: 107,150.8 kg; Campaign: 1,298,790.5 kg
Network model is valid
Network model is valid
TEST 3.5 - PASS: Apollo 17: more lunar equipment demand raises the objective, less lowers it
        objective: 94,783.5 (320 kg) < 107,150.8 (420 kg) < 107,642.0 (520 kg)


### ISRU on the whole mission (Campaign)

In [15]:
@test("3.6", "ISRU eligibility and limits: active ISRU only on lunar-surface holdovers, never above max mass, never in the 0-400 kg dead zone")
def _():
    m, ctx = SOLVED["Campaign"]
    com, isru = ctx["Commodities"], ctx["isru_config"]
    active = com.isru_indices["active"]
    deployed = []
    for t, i, j, a in all_arcs(ctx):
        for v in range(ctx["V"]):
            mass = ctx["x_outflow"][v][i][j][t][active][0].X
            if mass > 1e-6:
                assert cc.is_eligible_isru_arc(i, j, isru), f"active ISRU ({mass:.1f} kg) on non-eligible arc {(t, i, j)}, v={v}"
                assert mass <= isru.max_mass + 1e-6, f"active ISRU {mass:.1f} kg exceeds max mass {isru.max_mass}"
                assert mass >= 400 - 1e-6, f"active ISRU {mass:.1f} kg is in the dead zone (< 400 kg) on arc {(t, i, j)}"
                deployed.append(mass)
    assert deployed, "no active ISRU deployed at all"
    return f"{len(deployed)} lunar holdover arcs carry active ISRU, between {min(deployed):.0f} and {max(deployed):.0f} kg"


@test("3.7", "ISRU on vs off (Campaign): objective with ISRU is not higher than without it")
def _():
    on, off = SOLVED["Campaign"][0], SOLVED["Campaign, ISRU off"][0]
    assert on.Status == GRB.OPTIMAL and off.Status == GRB.OPTIMAL, f"status {on.Status}, {off.Status}"
    assert on.ObjVal <= off.ObjVal * (1 + 1e-6), f"ISRU on {on.ObjVal:,.1f} kg > ISRU off {off.ObjVal:,.1f} kg"
    saving = off.ObjVal - on.ObjVal
    return f"launched mass {off.ObjVal:,.0f} kg (off) -> {on.ObjVal:,.0f} kg (on): saving {saving:,.0f} kg ({100 * saving / off.ObjVal:.1f} %)"

TEST 3.6 - PASS: ISRU eligibility and limits: active ISRU only on lunar-surface holdovers, never above max mass, never in the 0-400 kg dead zone
        5 lunar holdover arcs carry active ISRU, between 504 and 1343 kg
TEST 3.7 - PASS: ISRU on vs off (Campaign): objective with ISRU is not higher than without it
        launched mass 1,337,334 kg (off) -> 1,298,791 kg (on): saving 38,543 kg (2.9 %)
